# Week 14 Portfolio Milestone 3: Troubleshoot a SQL–Python Pipeline

**Student Version**

Week 13 asked you to create a reproducible SQL→Pandas→analysis→visualization pipeline. This milestone gives you a controlled version of that pipeline with five silent logic defects. Diagnose each defect with control totals, correct the smallest relevant section, and document evidence that the correction worked.

**Estimated time:** 2.5–3 hours  
**AI use:** Level 3. AI may explain an error or draft a focused correction, but every suggestion must be verified. Do not use real patient or workplace data.


## Workflow

1. Read <code>Week14_Reading_Troubleshooting_SQL_Python_Pipelines.md</code>.
2. Save a copy of this notebook in Google Drive.
3. Run setup and baseline cells.
4. Repair Tickets 1–5 in order. Cells marked TODO require student work.
5. Complete the troubleshooting log and AI-use record.
6. Restart the runtime and run all cells from top to bottom.
7. Set <code>RUN_FINAL_GATE = True</code> only when every ticket passes.

The synthetic practice database is rebuilt by the notebook; no external data file is required.


## Evaluation (20 points)

| Criterion | Points |
|---|---:|
| Ticket 1: referral grain | 3 |
| Ticket 2: unmatched/NULL follow-up records | 3 |
| Ticket 3: clinic denominators | 3 |
| Ticket 4: date arithmetic | 3 |
| Ticket 5: chart alignment | 2 |
| Troubleshooting log | 3 |
| Clean run and regression suite | 2 |
| Interpretation, readiness, AI verification | 1 |


## Setup: rebuild the controlled practice database


In [ ]:
from pathlib import Path
import sqlite3
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

def display(value):
    if hasattr(value, "to_string"):
        print(value.to_string(index=False))
    else:
        print(value)

WORK_DIR = Path("/content/week14_troubleshooting") if Path("/content").exists() else Path(tempfile.gettempdir()) / "week14_troubleshooting"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "community_followup_week14.sqlite"

CONTROL_TOTALS = {
    "clinics": 3, "patients": 9, "encounters": 11, "referrals": 8,
    "followups": 6, "pending_referrals": 4, "completed_referrals": 4,
    "median_completed_followup_days": 7.5,
}
EXPECTED_RATES = {
    "Northside Clinic": 66.7,
    "Riverside Clinic": 33.3,
    "Valley Clinic": 50.0,
}

SCHEMA_SQL = """
PRAGMA foreign_keys = ON;
CREATE TABLE clinics (
 clinic_id INTEGER PRIMARY KEY,
 clinic_name TEXT NOT NULL UNIQUE
);
CREATE TABLE patients (
 patient_id INTEGER PRIMARY KEY,
 clinic_id INTEGER NOT NULL,
 age_group TEXT NOT NULL,
 social_risk TEXT NOT NULL CHECK (social_risk IN ('Low','Moderate','High')),
 FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id)
);
CREATE TABLE encounters (
 encounter_id INTEGER PRIMARY KEY,
 patient_id INTEGER NOT NULL,
 encounter_date TEXT NOT NULL,
 FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);
CREATE TABLE referrals (
 referral_id INTEGER PRIMARY KEY,
 encounter_id INTEGER NOT NULL UNIQUE,
 referral_date TEXT NOT NULL,
 service TEXT NOT NULL,
 priority TEXT NOT NULL CHECK (priority IN ('Routine','Urgent')),
 FOREIGN KEY (encounter_id) REFERENCES encounters(encounter_id)
);
CREATE TABLE followups (
 followup_id INTEGER PRIMARY KEY,
 referral_id INTEGER NOT NULL UNIQUE,
 followup_date TEXT NOT NULL,
 completed INTEGER NOT NULL CHECK (completed IN (0,1)),
 FOREIGN KEY (referral_id) REFERENCES referrals(referral_id)
);
"""

CLINICS = [(1,"Northside Clinic"),(2,"Riverside Clinic"),(3,"Valley Clinic")]
PATIENTS = [
 (1,1,"65-74","High"),(2,1,"45-64","Moderate"),(3,1,"75+","Low"),
 (4,2,"45-64","High"),(5,2,"65-74","Moderate"),(6,2,"18-44","Low"),
 (7,3,"75+","High"),(8,3,"65-74","Moderate"),(9,3,"45-64","Low")
]
ENCOUNTERS = [
 (101,1,"2026-09-01"),(102,1,"2026-09-20"),(103,2,"2026-09-05"),
 (104,3,"2026-09-08"),(105,4,"2026-09-02"),(106,4,"2026-09-25"),
 (107,5,"2026-09-07"),(108,6,"2026-09-09"),(109,7,"2026-09-03"),
 (110,8,"2026-09-11"),(111,9,"2026-09-12")
]
REFERRALS = [
 (201,102,"2026-09-20","Care coordination","Urgent"),
 (202,103,"2026-09-05","Medication review","Routine"),
 (203,104,"2026-09-08","Transportation","Routine"),
 (204,106,"2026-09-25","Care coordination","Urgent"),
 (205,107,"2026-09-07","Medication review","Routine"),
 (206,108,"2026-09-09","Transportation","Routine"),
 (207,109,"2026-09-03","Care coordination","Urgent"),
 (208,110,"2026-09-11","Medication review","Routine")
]
FOLLOWUPS = [
 (301,201,"2026-09-27",1),(302,202,"2026-09-12",1),
 (303,204,"2026-10-03",1),(304,205,"2026-09-14",0),
 (305,207,"2026-09-17",1),(306,208,"2026-10-01",0)
]

def rebuild_database(db_path=DB_PATH):
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    con.executescript(SCHEMA_SQL)
    con.executemany("INSERT INTO clinics VALUES (?,?)", CLINICS)
    con.executemany("INSERT INTO patients VALUES (?,?,?,?)", PATIENTS)
    con.executemany("INSERT INTO encounters VALUES (?,?,?)", ENCOUNTERS)
    con.executemany("INSERT INTO referrals VALUES (?,?,?,?,?)", REFERRALS)
    con.executemany("INSERT INTO followups VALUES (?,?,?,?)", FOLLOWUPS)
    con.commit()
    con.execute("PRAGMA foreign_keys = ON")
    return con

connection = rebuild_database()
print("Fresh practice database created:", DB_PATH.name)


## Baseline: confirm table counts and foreign-key integrity


In [ ]:
table_counts = pd.read_sql_query(
    """
    SELECT 'clinics' AS table_name, COUNT(*) AS row_count FROM clinics
    UNION ALL SELECT 'patients', COUNT(*) FROM patients
    UNION ALL SELECT 'encounters', COUNT(*) FROM encounters
    UNION ALL SELECT 'referrals', COUNT(*) FROM referrals
    UNION ALL SELECT 'followups', COUNT(*) FROM followups
    """, connection
)
display(table_counts)
foreign_key_problems = connection.execute("PRAGMA foreign_key_check").fetchall()
counts_match = all(int(row.row_count) == CONTROL_TOTALS[row.table_name] for row in table_counts.itertuples())
print("Foreign-key problems:", foreign_key_problems)
print("Baseline table counts match:", counts_match)


## Ticket 1 — A join inflates referral counts

**Expected grain:** one row per referral before clinic summarization.  
**Control total:** 8 referrals.


In [ ]:
# TODO 1: Repair the query while preserving one row per referral.
ticket1_sql = """
SELECT c.clinic_name, COUNT(*) AS referral_count
FROM clinics AS c
JOIN patients AS p ON p.clinic_id = c.clinic_id
JOIN encounters AS e ON e.patient_id = p.patient_id
JOIN referrals AS r ON r.encounter_id IN (
    SELECT e2.encounter_id FROM encounters AS e2
    WHERE e2.patient_id = p.patient_id
)
GROUP BY c.clinic_name
ORDER BY c.clinic_name;
"""
ticket1_df = pd.read_sql_query(ticket1_sql, connection)
display(ticket1_df)
ticket1_total = int(ticket1_df["referral_count"].sum())
print("Observed referrals:", ticket1_total)
print("Expected referrals:", CONTROL_TOTALS["referrals"])
print("PASS" if ticket1_total == CONTROL_TOTALS["referrals"] else "FAIL — investigate join grain")


## Ticket 2 — Pending referrals disappear

Pending means there is no completed follow-up, including an attempted-but-incomplete follow-up and no follow-up row. **Control total:** 4.


In [ ]:
# TODO 2: Retain referrals that have no follow-up row.
ticket2_sql = """
SELECT r.referral_id, c.clinic_name, f.completed
FROM referrals AS r
JOIN encounters AS e ON e.encounter_id = r.encounter_id
JOIN patients AS p ON p.patient_id = e.patient_id
JOIN clinics AS c ON c.clinic_id = p.clinic_id
LEFT JOIN followups AS f ON f.referral_id = r.referral_id
WHERE f.completed = 0
ORDER BY r.referral_id;
"""
ticket2_df = pd.read_sql_query(ticket2_sql, connection)
display(ticket2_df)
ticket2_total = len(ticket2_df)
print("Observed pending referrals:", ticket2_total)
print("Expected pending referrals:", CONTROL_TOTALS["pending_referrals"])
print("PASS" if ticket2_total == CONTROL_TOTALS["pending_referrals"] else "FAIL — inspect NULL handling")


## Validated referral-level extract — do not change


In [ ]:
referral_level_sql = """
SELECT r.referral_id, c.clinic_name, r.referral_date,
       COALESCE(f.completed,0) AS completed, f.followup_date
FROM referrals AS r
JOIN encounters AS e ON e.encounter_id = r.encounter_id
JOIN patients AS p ON p.patient_id = e.patient_id
JOIN clinics AS c ON c.clinic_id = p.clinic_id
LEFT JOIN followups AS f ON f.referral_id = r.referral_id
ORDER BY r.referral_id;
"""
referral_level_df = pd.read_sql_query(referral_level_sql, connection)
display(referral_level_df)
print("One row per referral:", referral_level_df["referral_id"].is_unique)


## Ticket 3 — Rates use the wrong denominator

**Control rates:** Northside 66.7%, Riverside 33.3%, Valley 50.0%.


In [ ]:
# TODO 3: Use each clinic own referral total as its denominator.
def calculate_completion_rates(df):
    summary = df.groupby("clinic_name", as_index=False).agg(completed=("completed","sum"))
    summary["completion_rate_pct"] = (summary["completed"] / len(df) * 100).round(1)
    return summary

rate_df = calculate_completion_rates(referral_level_df)
display(rate_df)
observed_rates = dict(zip(rate_df["clinic_name"], rate_df["completion_rate_pct"]))
print("Observed rates:", observed_rates)
print("Expected rates:", EXPECTED_RATES)
print("PASS" if observed_rates == EXPECTED_RATES else "FAIL — inspect denominator grain")


## Ticket 4 — Date arithmetic fails across month boundaries

**Control results:** no negative intervals; median completed follow-up time = 7.5 days.


In [ ]:
# TODO 4: Replace day-of-month subtraction with complete date arithmetic.
def calculate_completed_followup_days(df):
    completed = df.loc[df["completed"] == 1].copy()
    completed["days_to_followup"] = (
        completed["followup_date"].str[-2:].astype(int)
        - completed["referral_date"].str[-2:].astype(int)
    )
    return completed

interval_df = calculate_completed_followup_days(referral_level_df)
display(interval_df[["referral_id","referral_date","followup_date","days_to_followup"]])
negative_intervals = int((interval_df["days_to_followup"] < 0).sum())
median_days = float(interval_df["days_to_followup"].median())
print("Negative intervals:", negative_intervals)
print("Observed median days:", median_days)
print("Expected median days:", CONTROL_TOTALS["median_completed_followup_days"])
print("PASS" if negative_intervals == 0 and median_days == CONTROL_TOTALS["median_completed_followup_days"] else "FAIL — inspect date types")


## Ticket 5 — The chart answers the wrong question

Repair the plotted values, title, and axis label so the chart compares completion rates.


In [ ]:
# TODO 5: Plot completion rates and repair the title and y-axis units.
def make_completion_chart(summary_df):
    plot_values = summary_df["completed"]
    fig, ax = plt.subplots(figsize=(8,4.5))
    ax.bar(summary_df["clinic_name"], plot_values, color="#4472C4")
    ax.set_title("Completed referrals by clinic")
    ax.set_ylabel("Completed referrals")
    ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    return fig, ax, list(plot_values.round(1))

fig, ax, chart_values = make_completion_chart(rate_df)
plt.show()
expected_chart_values = [EXPECTED_RATES[name] for name in rate_df["clinic_name"]]
print("Values plotted:", chart_values)
print("Expected rate values:", expected_chart_values)
print("PASS" if chart_values == expected_chart_values else "FAIL — chart answers the wrong question")


## Troubleshooting and validation log


| Ticket | Symptom | Hypothesis/root cause | Focused test | Correction | Retest evidence |
|---|---|---|---|---|---|
| 1 | [complete] | [complete] | [complete] | [complete] | [complete] |
| 2 | [complete] | [complete] | [complete] | [complete] | [complete] |
| 3 | [complete] | [complete] | [complete] | [complete] | [complete] |
| 4 | [complete] | [complete] | [complete] | [complete] | [complete] |
| 5 | [complete] | [complete] | [complete] | [complete] | [complete] |


## Final clean-run regression gate


In [ ]:
# TODO: Change to True only after all five tickets pass.
RUN_FINAL_GATE = False
checks = {
 "foreign_keys": foreign_key_problems == [],
 "referral_grain": ticket1_total == CONTROL_TOTALS["referrals"],
 "pending_logic": ticket2_total == CONTROL_TOTALS["pending_referrals"],
 "clinic_rates": observed_rates == EXPECTED_RATES,
 "date_intervals": negative_intervals == 0 and median_days == CONTROL_TOTALS["median_completed_followup_days"],
 "chart_metric": chart_values == expected_chart_values,
}
validation_df = pd.DataFrame([{"check":k,"passed":v} for k,v in checks.items()])
display(validation_df)
if RUN_FINAL_GATE:
    assert validation_df["passed"].all(), "One or more regression checks failed."
    print("FINAL GATE PASSED — the repaired pipeline is ready for submission.")
else:
    print("Final gate not run. Repair all tickets, set RUN_FINAL_GATE = True, and rerun.")


## Interpretation and readiness reflection

Respond in 3–5 sentences:

1. Which defect was most likely to produce a believable but incorrect conclusion, and why?
2. What validation check will you add to your Week 13 pipeline?
3. What remains before your portfolio workflow is dependable?


## AI-use and verification record

- **Prompt/question:** [complete or state “No AI used”]
- **Suggestion summarized:** [complete]
- **Verification test:** [complete]
- **Decision and rationale:** [complete]

## Submission

Restart and run all cells, confirm the final gate passes, and download the executed notebook as <code>Week14_Troubleshoot_SQL_Python_Pipeline_Completed.ipynb</code>. Add it to the Week 13 GitHub portfolio repository. Submit the notebook and repository link in Canvas.
